In [1]:
import torch

In [2]:
def generate_merge_masks(lens, T, prob=0.15, max_span=5):

    B = len(lens)

    start_mask = torch.zeros(B, T, dtype=torch.bool)
    pad_mask = torch.zeros(B, T, dtype=torch.bool)

    for b in range(B):

        t = 0
        L = lens[b]

        while t < L-1:

            if torch.rand(1).item() < prob:

                span_len = torch.randint(2, max_span+1, (1,)).item()
                span_len = min(span_len, L - t)

                start_mask[b, t] = True

                if span_len > 1:
                    pad_mask[b, t+1:t+span_len] = True

                t += span_len

            else:
                t += 1

    return start_mask, pad_mask

In [24]:
# Create random input
lens = [10, 8, 12]  # sequence lengths for batch of 3
T = 15  # maximum sequence length
B = len(lens)
lens= torch.tensor(lens)
# Create random X
X = torch.randint(0, 10, (B, T, 1))
valid_mask = torch.arange(T).unsqueeze(0) < lens.unsqueeze(1)
X[~valid_mask] = 0
# Apply the function
start_mask, pad_mask = generate_merge_masks(lens, T, prob=0.15, max_span=5)

print("Start mask shape:", start_mask.shape)
print("Pad mask shape:", pad_mask.shape)
print("Start mask:\n", start_mask)
print("Pad mask:\n", pad_mask)

Start mask shape: torch.Size([3, 15])
Pad mask shape: torch.Size([3, 15])
Start mask:
 tensor([[ True, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False],
        [False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False],
        [ True, False, False, False,  True, False, False, False, False, False,
         False, False, False, False, False]])
Pad mask:
 tensor([[False,  True,  True,  True, False, False, False, False, False, False,
         False, False, False, False, False],
        [False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False],
        [False,  True,  True, False, False,  True,  True,  True,  True, False,
         False, False, False, False, False]])


In [28]:
def apply_merge(x, start_mask, pad_mask, pad_token):

    B, T, _ = x.shape

    span_mask = start_mask | pad_mask
    span_mask_f = span_mask.float().unsqueeze(-1)

    summed = (x * span_mask_f).sum(dim=1)
    counts = span_mask_f.sum(dim=1).clamp(min=1e-6)

    merged = summed / counts

    merged_expand = merged.unsqueeze(1).expand(-1, T, -1)

    x_aug = torch.where(start_mask.unsqueeze(-1), merged_expand, x)
    x_aug = torch.where(pad_mask.unsqueeze(-1), pad_token, x_aug)

    return x_aug

In [29]:
X_aug = apply_merge(X, start_mask, pad_mask, torch.zeros(1))

In [30]:
print(X_aug.shape)
print(X.shape)

print(X_aug[0])
print(X_aug[0].shape)
print(X[0])

torch.Size([3, 15, 1])
torch.Size([3, 15, 1])
tensor([[5.5000],
        [0.0000],
        [0.0000],
        [0.0000],
        [9.0000],
        [8.0000],
        [5.0000],
        [1.0000],
        [3.0000],
        [3.0000],
        [0.0000],
        [0.0000],
        [0.0000],
        [0.0000],
        [0.0000]])
torch.Size([15, 1])
tensor([[4],
        [6],
        [4],
        [8],
        [9],
        [8],
        [5],
        [1],
        [3],
        [3],
        [0],
        [0],
        [0],
        [0],
        [0]])
